# Notebook 7: End-to-End Demo & Presentation Visualizations
## Interactive Pipeline Demo + Publication-Ready Figures

This notebook provides:
1. **End-to-end demo**: Input any resume text + job description → get full gap analysis
2. **Presentation-ready visualizations** for the final report
3. **Summary dashboard** of all system metrics


In [2]:
from google.colab import drive
import os

# 1. Mount the Drive
drive.mount('/content/drive')

# 2. Define your actual Drive project path
drive_path = '/content/drive/MyDrive/Colab Notebooks/DM_project_1'

# 3. Create the symbolic links
# This links the 'data', 'models', and 'outputs' folders from Drive to your local Colab env
folders = ['data', 'models', 'outputs']

for folder in folders:
    source = os.path.join(drive_path, folder)
    link = f'/content/{folder}'

    # Remove existing local folder/link if it exists to avoid errors
    if os.path.exists(link) or os.path.islink(link):
        !rm -rf {link}

    # Create the link (Shortcut)
    !ln -s "{source}" "{link}"
    print(f"✅ Linked local '{folder}/' to Drive '{folder}/'")

Mounted at /content/drive
✅ Linked local 'data/' to Drive 'data/'
✅ Linked local 'models/' to Drive 'models/'
✅ Linked local 'outputs/' to Drive 'outputs/'


In [3]:
# ============================================================
# Setup
# ============================================================
import os
import re
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import spacy
import warnings
warnings.filterwarnings('ignore')

# Load all models and data
nlp = spacy.load('en_core_web_sm', disable=['ner', 'parser'])

with open('data/processed/skill_lookup.json') as f:
    skill_lookup = json.load(f)
with open('data/processed/skill_category_map.json') as f:
    skill_category_map = json.load(f)
with open('data/processed/extraction_results.pkl', 'rb') as f:
    extraction_data = pickle.load(f)
with open('data/processed/embeddings_data.pkl', 'rb') as f:
    emb_data = pickle.load(f)

df_jobs = pd.read_csv('data/processed/jobs_with_skills.csv')
df_resumes = pd.read_csv('data/processed/resumes_with_skills.csv')
df_jobs['skills_list'] = df_jobs['extracted_skills'].apply(json.loads)
df_resumes['skills_list'] = df_resumes['extracted_skills'].apply(json.loads)

with open('data/processed/column_config.json') as f:
    col_config = json.load(f)

# Load models
with open('models/skill_gap_analyzer.pkl', 'rb') as f:
    analyzer = pickle.load(f)

from sentence_transformers import SentenceTransformer
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

print("✅ All models and data loaded!")


FileNotFoundError: [Errno 2] No such file or directory: 'models/skill_gap_analyzer.pkl'

## 1. End-to-End Demo Pipeline

Paste any resume and job description below to see the full multi-agent system in action.


In [ ]:
# ============================================================
# 1. Skill Extractor (from Notebook 2)
# ============================================================

class SkillExtractor:
    def __init__(self, skill_lookup, skill_category_map):
        self.skill_lookup = skill_lookup
        self.skill_category_map = skill_category_map
        self.sorted_aliases = sorted(skill_lookup.keys(), key=len, reverse=True)
        self.patterns = []
        for alias in self.sorted_aliases:
            pattern = re.compile(r'\b' + re.escape(alias) + r'\b', re.IGNORECASE)
            self.patterns.append((pattern, alias))

    def extract_skills(self, text):
        if not isinstance(text, str) or len(text) < 5:
            return {'skills': [], 'skill_counts': {}, 'skill_categories': {}, 'matched_phrases': []}
        text_lower = text.lower()
        found_skills = Counter()
        matched_phrases = []
        matched_positions = set()
        for pattern, alias in self.patterns:
            for match in pattern.finditer(text_lower):
                start, end = match.start(), match.end()
                if any(pos in matched_positions for pos in range(start, end)):
                    continue
                canonical = self.skill_lookup[alias]
                found_skills[canonical] += 1
                matched_phrases.append((match.group(), canonical))
                matched_positions.update(range(start, end))
        skills_list = list(found_skills.keys())
        categories = {s: self.skill_category_map.get(s, 'unknown') for s in skills_list}
        return {'skills': skills_list, 'skill_counts': dict(found_skills),
                'skill_categories': categories, 'matched_phrases': matched_phrases}

extractor = SkillExtractor(skill_lookup, skill_category_map)
print("✅ Extractor ready")


In [ ]:
# ============================================================
# 2. Full Demo: Input Resume + Job Description
# ============================================================

# ╔══════════════════════════════════════════════════════════╗
# ║  EDIT THESE TO TRY YOUR OWN RESUME AND JOB DESCRIPTION  ║
# ╚══════════════════════════════════════════════════════════╝

DEMO_RESUME = """
Experienced data analyst with 3 years of experience in Python, SQL, and Excel.
Built interactive dashboards using Tableau and Power BI for executive reporting.
Strong background in statistical analysis, A/B testing, and data visualization.
Experience with pandas, numpy, and matplotlib for data manipulation and visualization.
Familiar with basic machine learning concepts using scikit-learn.
Worked with MySQL and PostgreSQL databases. Comfortable with Git version control.
Strong communication skills with experience presenting to stakeholders.
Bachelor's degree in Statistics from ASU.
"""

DEMO_JOB = """
Senior Data Scientist - Machine Learning
We are looking for a Senior Data Scientist to join our ML team.

Requirements:
- 5+ years of experience with Python and SQL
- Strong expertise in machine learning and deep learning
- Proficiency with TensorFlow or PyTorch for model development
- Experience with NLP or computer vision projects
- Knowledge of cloud platforms (AWS or GCP)
- Experience with Docker and Kubernetes for model deployment
- Strong skills in statistical analysis and A/B testing
- Experience with Apache Spark or similar big data tools
- Excellent communication and presentation skills
- MS or PhD in Computer Science, Statistics, or related field
"""

print("=" * 80)
print("MULTI-AGENT RESUME SKILL GAP ANALYSIS DEMO")
print("=" * 80)

# Agent 1: Extract resume skills
print("\n▶ Agent 1: Extracting resume skills...")
resume_result = extractor.extract_skills(DEMO_RESUME)
resume_skills = resume_result['skills']
print(f"  Found {len(resume_skills)} skills: {', '.join(resume_skills)}")

# Agent 2: Extract job skills
print("\n▶ Agent 2: Extracting job requirements...")
job_result = extractor.extract_skills(DEMO_JOB)
job_skills = job_result['skills']
print(f"  Found {len(job_skills)} skills: {', '.join(job_skills)}")

# Agent 3: Gap analysis
print("\n▶ Agent 3: Performing skill gap analysis...")
gap_result = analyzer.analyze(resume_skills, job_skills, "Senior Data Scientist - ML")

print(f"\n{'━'*80}")
print(f"  SKILL GAP ANALYSIS REPORT")
print(f"{'━'*80}")
print(f"  Target role: {gap_result['job_title']}")
print(f"  Total required skills: {gap_result['total_job_skills']}")
print(f"  Exact matches: {gap_result['num_matched']} ({gap_result['coverage_rate']:.0%})")
print(f"  Partial matches: {gap_result['num_partial']} ({gap_result['partial_coverage_rate']:.0%})")
print(f"  Missing: {gap_result['num_missing']}")
print(f"  Overall coverage: {gap_result['total_coverage_rate']:.0%}")

print(f"\n  ✅ MATCHED SKILLS:")
for m in gap_result['matched']:
    print(f"     • {m['skill']} [{m['category']}]")

print(f"\n  🔶 PARTIAL MATCHES:")
for p in gap_result['partial_matches']:
    print(f"     • {p['job_skill']} ← your '{p['resume_match']}' (similarity: {p['similarity']:.2f})")

print(f"\n  ❌ MISSING SKILLS (ranked by priority):")
for m in gap_result['missing']:
    bar = '█' * int(m['priority_score'] * 20)
    print(f"     • {m['skill']} [{m['category']}]")
    print(f"       Priority: {bar} {m['priority_score']:.2f}")
    if m['closest_resume_skill']:
        print(f"       Closest skill you have: {m['closest_resume_skill']} "
              f"(sim: {m['closest_similarity']:.2f})")

print(f"\n  💡 RECOMMENDATIONS:")
for r in gap_result['recommendations']:
    icon = {'high': '🔴', 'medium': '🟡', 'info': '🟢'}.get(r['priority'], '⚪')
    print(f"     {icon} {r['message']}")


In [ ]:
# ============================================================
# 3. Visual Gap Report (Presentation-Ready)
# ============================================================

fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.3)

# 3a. Donut chart: match breakdown
ax1 = fig.add_subplot(gs[0, 0])
sizes = [gap_result['num_matched'], gap_result['num_partial'], gap_result['num_missing']]
labels = ['Matched', 'Partial', 'Missing']
colors = ['#0F6E56', '#EF9F27', '#D85A30']
wedges, texts, autotexts = ax1.pie(sizes, labels=labels, colors=colors,
                                    autopct='%1.0f%%', startangle=90,
                                    pctdistance=0.75, wedgeprops=dict(width=0.4))
ax1.set_title('Skill Match Breakdown', fontweight='bold', fontsize=12)

# 3b. Bar chart: missing skills by priority
ax2 = fig.add_subplot(gs[0, 1:])
missing_sorted = sorted(gap_result['missing'], key=lambda x: x['priority_score'], reverse=True)
skill_names = [m['skill'] for m in missing_sorted[:8]]
priorities = [m['priority_score'] for m in missing_sorted[:8]]
colors_bar = ['#E24B4A' if p >= 0.7 else '#EF9F27' if p >= 0.5 else '#888780' for p in priorities]
bars = ax2.barh(range(len(skill_names)), priorities, color=colors_bar, alpha=0.85)
ax2.set_yticks(range(len(skill_names)))
ax2.set_yticklabels(skill_names)
ax2.set_xlabel('Priority Score')
ax2.set_title('Missing Skills by Priority', fontweight='bold', fontsize=12)
ax2.invert_yaxis()
ax2.set_xlim(0, 1)

# Add category labels
for i, m in enumerate(missing_sorted[:8]):
    ax2.text(priorities[i] + 0.02, i, f"[{m['category'][:12]}]",
             va='center', fontsize=8, color='gray')

# 3c. Category coverage radar-style comparison
ax3 = fig.add_subplot(gs[1, 0:2])
resume_cats = Counter()
job_cats = Counter()
for s in resume_skills:
    cat = skill_category_map.get(s, 'unknown')
    resume_cats[cat] += 1
for s in job_skills:
    cat = skill_category_map.get(s, 'unknown')
    job_cats[cat] += 1

all_cats = sorted(set(list(resume_cats.keys()) + list(job_cats.keys())))
x_pos = np.arange(len(all_cats))
width = 0.35
ax3.bar(x_pos - width/2, [resume_cats.get(c, 0) for c in all_cats],
        width, label='Your Skills', color='#534AB7', alpha=0.8)
ax3.bar(x_pos + width/2, [job_cats.get(c, 0) for c in all_cats],
        width, label='Required', color='#D85A30', alpha=0.8)
ax3.set_xticks(x_pos)
ax3.set_xticklabels([c.replace('_', ' ').title()[:15] for c in all_cats],
                    rotation=45, ha='right', fontsize=9)
ax3.set_ylabel('Number of Skills')
ax3.set_title('Category-Level Comparison', fontweight='bold', fontsize=12)
ax3.legend()

# 3d. Summary metrics
ax4 = fig.add_subplot(gs[1, 2])
ax4.axis('off')
metrics_text = f"""
  Overall Coverage
  ─────────────────
  Exact:   {gap_result['coverage_rate']:.0%}
  Partial: {gap_result['partial_coverage_rate']:.0%}
  Total:   {gap_result['total_coverage_rate']:.0%}

  Skills Summary
  ─────────────────
  Resume skills: {len(resume_skills)}
  Job requires:  {len(job_skills)}
  Matched:       {gap_result['num_matched']}
  Gaps:          {gap_result['num_missing']}
"""
ax4.text(0.1, 0.95, metrics_text, transform=ax4.transAxes,
         fontsize=11, verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='#F1EFE8', alpha=0.8))

plt.suptitle('Multi-Agent Skill Gap Analysis Report', fontsize=16, fontweight='bold', y=1.02)
plt.savefig('outputs/07_demo_gap_report.png', dpi=150, bbox_inches='tight')
plt.show()


## 2. Presentation-Ready System Visualizations


In [ ]:
# ============================================================
# 4. Dataset Summary Visualization
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

cat_col = col_config['resume_cat_col']
title_col = col_config['jobs_title_col']

# 4a. Dataset sizes
datasets = ['LinkedIn\nJobs', 'Resumes']
sizes = [len(df_jobs), len(df_resumes)]
colors = ['#534AB7', '#0F6E56']
axes[0].bar(datasets, sizes, color=colors, alpha=0.8, width=0.5)
for i, v in enumerate(sizes):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')
axes[0].set_title('Dataset Sizes', fontweight='bold')
axes[0].set_ylabel('Number of Documents')

# 4b. Skill extraction summary
all_resume_skills = set()
for s in df_resumes['skills_list']:
    all_resume_skills.update(s)
all_job_skills = set()
for s in df_jobs['skills_list']:
    all_job_skills.update(s)

categories_data = ['Resume\nSkills', 'Job\nSkills', 'Overlap']
values = [len(all_resume_skills), len(all_job_skills), len(all_resume_skills & all_job_skills)]
axes[1].bar(categories_data, values, color=['#0F6E56', '#534AB7', '#D85A30'], alpha=0.8, width=0.5)
for i, v in enumerate(values):
    axes[1].text(i, v + 1, str(v), ha='center', fontweight='bold')
axes[1].set_title('Extracted Skill Vocabulary', fontweight='bold')
axes[1].set_ylabel('Unique Skills')

# 4c. Avg skills per document
doc_types = ['Per Resume', 'Per Job']
avg_skills = [df_resumes['skills_list'].apply(len).mean(),
              df_jobs['skills_list'].apply(len).mean()]
std_skills = [df_resumes['skills_list'].apply(len).std(),
              df_jobs['skills_list'].apply(len).std()]
axes[2].bar(doc_types, avg_skills, yerr=std_skills, color=['#0F6E56', '#534AB7'],
            alpha=0.8, width=0.5, capsize=5)
for i, v in enumerate(avg_skills):
    axes[2].text(i, v + std_skills[i] + 0.3, f"{v:.1f}", ha='center', fontweight='bold')
axes[2].set_title('Average Skills Extracted', fontweight='bold')
axes[2].set_ylabel('Skills per Document')

plt.tight_layout()
plt.savefig('outputs/07_dataset_summary.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ============================================================
# 5. Results Dashboard (All Metrics in One Figure)
# ============================================================

try:
    with open('data/processed/gap_analysis_summary.csv') as f:
        analysis_df = pd.read_csv(f)
    with open('data/processed/baseline_results.json') as f:
        baseline_results = json.load(f)
    has_baseline = True
except:
    has_baseline = False

fig = plt.figure(figsize=(18, 10))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.3)

# 5a. Coverage distribution
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(analysis_df['total_coverage_rate'], bins=25, color='#534AB7',
         alpha=0.7, edgecolor='white')
ax1.axvline(analysis_df['total_coverage_rate'].mean(), color='red', linestyle='--',
            label=f"Mean: {analysis_df['total_coverage_rate'].mean():.2f}")
ax1.set_title('Total Coverage Distribution', fontweight='bold')
ax1.set_xlabel('Coverage Rate')
ax1.legend()

# 5b. Missing skills frequency
all_missing = []
for a in all_analyses:
    for m in a['missing']:
        all_missing.append(m['skill'])
missing_top = Counter(all_missing).most_common(10)
ax2 = fig.add_subplot(gs[0, 1])
ax2.barh([s[0] for s in missing_top], [s[1] for s in missing_top],
         color='#D85A30', alpha=0.8)
ax2.set_title('Most Common Skill Gaps', fontweight='bold')
ax2.invert_yaxis()

# 5c. Baseline comparison
ax3 = fig.add_subplot(gs[0, 2])
if has_baseline and baseline_results:
    models = list(baseline_results.keys())
    metrics = ['accuracy', 'f1', 'precision', 'recall']
    x = np.arange(len(metrics))
    width = 0.3
    for i, model in enumerate(models):
        values = [baseline_results[model][m] for m in metrics]
        ax3.bar(x + i*width, values, width, label=model,
                color=['#534AB7', '#0F6E56'][i], alpha=0.8)
    ax3.set_xticks(x + width/2)
    ax3.set_xticklabels(['Acc', 'F1', 'Prec', 'Recall'])
    ax3.set_ylim(0, 1)
    ax3.legend(fontsize=8)
ax3.set_title('Baseline Classifier Metrics', fontweight='bold')

# 5d. Coverage by similarity
ax4 = fig.add_subplot(gs[1, 0])
ax4.scatter(analysis_df['similarity_score'], analysis_df['total_coverage_rate'],
            alpha=0.2, s=8, color='#0F6E56')
z = np.polyfit(analysis_df['similarity_score'], analysis_df['total_coverage_rate'], 1)
p = np.poly1d(z)
x_line = np.linspace(analysis_df['similarity_score'].min(),
                      analysis_df['similarity_score'].max(), 100)
ax4.plot(x_line, p(x_line), 'r--', linewidth=2)
corr = analysis_df['similarity_score'].corr(analysis_df['total_coverage_rate'])
ax4.set_title(f'Coverage vs Similarity (r={corr:.3f})', fontweight='bold')
ax4.set_xlabel('Similarity Score')
ax4.set_ylabel('Coverage Rate')

# 5e. Match type breakdown by category
ax5 = fig.add_subplot(gs[1, 1:])
cat_stats = analysis_df.groupby('resume_category').agg({
    'num_matched': 'mean',
    'num_partial': 'mean',
    'num_missing': 'mean',
}).sort_values('num_matched', ascending=True)

cat_stats.plot(kind='barh', stacked=True, ax=ax5,
               color=['#0F6E56', '#EF9F27', '#D85A30'], alpha=0.8)
ax5.set_title('Avg Skills by Match Type per Category', fontweight='bold')
ax5.set_xlabel('Average Number of Skills')
ax5.legend(['Matched', 'Partial', 'Missing'], loc='lower right', fontsize=9)

plt.suptitle('Multi-Agent System — Results Dashboard', fontsize=16, fontweight='bold')
plt.savefig('outputs/07_results_dashboard.png', dpi=200, bbox_inches='tight')
plt.show()

print("\n✅ All presentation figures saved to outputs/")


In [ ]:
# ============================================================
# 6. Quick Usage Guide
# ============================================================

print("""
╔══════════════════════════════════════════════════════════════╗
║          HOW TO USE THIS SYSTEM                              ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  1. EDIT the DEMO_RESUME and DEMO_JOB variables above       ║
║     with your own text                                       ║
║                                                              ║
║  2. RUN all cells to get:                                    ║
║     • Extracted skills from both documents                   ║
║     • Skill gap classification (matched/partial/missing)     ║
║     • Priority-ranked missing skills                         ║
║     • Actionable career recommendations                     ║
║     • Visual gap analysis report                             ║
║                                                              ║
║  3. CHECK the outputs/ folder for saved figures              ║
║                                                              ║
║  NOTEBOOKS:                                                  ║
║  01: Data preprocessing & skill dictionary                   ║
║  02: Skill extraction agents                                 ║
║  03: Embeddings & similarity computation                     ║
║  04: Skill gap analysis & recommendations                    ║
║  05: Baseline classifiers (kNN & Decision Tree)              ║
║  06: Comprehensive evaluation suite                          ║
║  07: This demo & visualizations (YOU ARE HERE)               ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")

print("\nPresentation figures saved:")
for f in sorted(os.listdir('outputs')):
    if f.endswith('.png'):
        size_kb = os.path.getsize(f'outputs/{f}') / 1024
        print(f"  📊 outputs/{f} ({size_kb:.0f} KB)")
